In [23]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors,AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
import rdkit.Chem.Descriptors3D
# from morfeus import xtb, read_xyz
import os

In [24]:
file_path = './MixtureSolDB_with_pure_only_complete.csv'

solubility_df = pd.read_csv(file_path)

# Добавляем дескрипторы из SMILES

In [25]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

def calc_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    desc = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'RingCount': Descriptors.RingCount(mol),
        'NumAromaticRings': Descriptors.NumAromaticRings(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'BertzCT': Descriptors.BertzCT(mol),
        'BalabanJ': Descriptors.BalabanJ(mol),
        'MaxPartialCharge': Descriptors.MaxPartialCharge(mol),
        'MinPartialCharge': Descriptors.MinPartialCharge(mol),
    }
    return desc

In [26]:
all_smiles = pd.concat([
    solubility_df['SMILES_Solute'],
    solubility_df['SMILES_Solvent1'],
    solubility_df['SMILES_Solvent2']
]).dropna().unique()

In [19]:
desc_list = []
valid_smiles = []

for sm in all_smiles:
    d = calc_rdkit_descriptors(sm)
    if d is not None:
        desc_list.append(d)
        valid_smiles.append(sm)

desc_df = pd.DataFrame(desc_list)
desc_df['SMILES'] = valid_smiles

In [27]:
desc_df

,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount,NumAromaticRings,FractionCSP3,HeavyAtomCount,BertzCT,BalabanJ,MaxPartialCharge,MinPartialCharge,SMILES
0,400.428,0.63710,147.84,4,7,7,1,1,0.500000,28,646.740613,6.210000e-07,0.414454,-0.479240,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...
1,108.144,0.85100,52.04,2,2,0,1,1,0.000000,8,166.388339,3.077660e+00,0.033414,-0.398667,Nc1cccc(N)c1
2,280.323,4.24720,39.44,0,3,3,3,3,0.166667,21,816.736205,2.120978e+00,0.199993,-0.490877,CC(C)Oc1ccc2c(=O)c(-c3ccccc3)coc2c1
3,368.297,2.01080,124.04,1,7,3,3,2,0.105263,27,1047.915446,2.306286e+00,0.335344,-0.477637,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...
4,153.572,1.61130,41.57,1,2,0,2,2,0.000000,10,354.520415,2.950090e+00,0.141716,-0.346007,Clc1ncnc2[nH]ccc12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,102.133,0.95790,26.30,0,2,1,0,0,0.800000,7,66.480406,3.126007e+00,0.302379,-0.463156,CC(=O)OC(C)C
666,88.150,1.16730,20.23,1,1,1,0,0,1.000000,6,33.693328,3.168490e+00,0.058870,-0.390483,CCC(C)(C)O
667,267.835,1.81390,0.00,0,0,0,0,0,1.000000,3,2.754888,1.632993e+00,0.051623,-0.074763,C(I)I
668,86.134,1.18690,9.23,0,1,0,1,0,1.000000,6,19.409910,2.000000e+00,0.046555,-0.381452,C1CCOCC1


In [29]:
# solute
solubility_df = solubility_df.merge(
    desc_df.add_prefix('solute_').rename(columns={'solute_SMILES': 'SMILES_Solute'}),
    on='SMILES_Solute',
    how='left'
)

# solvent1
solubility_df = solubility_df.merge(
    desc_df.add_prefix('S1_').rename(columns={'S1_SMILES': 'SMILES_Solvent1'}),
    on='SMILES_Solvent1',
    how='left'
)

# solvent2
solubility_df = solubility_df.merge(
    desc_df.add_prefix('S2_').rename(columns={'S2_SMILES': 'SMILES_Solvent2'}),
    on='SMILES_Solvent2',
    how='left'
)

In [32]:
solubility_df.columns

Index(['Compound_Name', 'SMILES_Solute', 'Temperature_K',
       'Solubility(mole_fraction)', 'LogS(mole_fraction)', 'Solvent1',
       'Solvent2', 'SMILES_Solvent1', 'SMILES_Solvent2', 'x1', 'x2',
       'IsPureSolventEndpoint', 'S1_pure_solubility', 'S1_pure_logS',
       'S1_temp_diff', 'S2_pure_solubility', 'S2_pure_logS', 'S2_temp_diff',
       'solute_MolWt', 'solute_MolLogP', 'solute_TPSA', 'solute_NumHDonors',
       'solute_NumHAcceptors', 'solute_NumRotatableBonds', 'solute_RingCount',
       'solute_NumAromaticRings', 'solute_FractionCSP3',
       'solute_HeavyAtomCount', 'solute_BertzCT', 'solute_BalabanJ',
       'solute_MaxPartialCharge', 'solute_MinPartialCharge', 'S1_MolWt',
       'S1_MolLogP', 'S1_TPSA', 'S1_NumHDonors', 'S1_NumHAcceptors',
       'S1_NumRotatableBonds', 'S1_RingCount', 'S1_NumAromaticRings',
       'S1_FractionCSP3', 'S1_HeavyAtomCount', 'S1_BertzCT', 'S1_BalabanJ',
       'S1_MaxPartialCharge', 'S1_MinPartialCharge', 'S2_MolWt', 'S2_MolLogP',
      

In [33]:
row = solubility_df.iloc[0]
print(np.log(row['Solubility(mole_fraction)']))
print(np.log10(row['Solubility(mole_fraction)']))
print(row['LogS(mole_fraction)'])

-2.439550948067203
-1.0594835150674329
-1.0594835150674329


In [34]:
solubility_df.to_csv("MixtureSolDB_with_features_small.csv", index=False)

# Другие искуственные фичи (не запускал)

In [9]:
df = solubility_df.copy()

# 1. эмпирическое предсказание
df['mix_logS_linear'] = (
    df['x1'] * df['S1_pure_logS'] +
    df['x2'] * df['S2_pure_logS']
)

# 2. простые композиционные признаки
df['x1_minus_x2'] = df['x1'] - df['x2']
df['x1_times_x2'] = df['x1'] * df['x2']
df['x1_squared'] = df['x1'] ** 2
df['x2_squared'] = df['x2'] ** 2

# 3. нелинейные признаки на чистых растворимостях
df['pure_logS_diff'] = df['S1_pure_logS'] - df['S2_pure_logS']
df['pure_logS_abs_diff'] = (df['S1_pure_logS'] - df['S2_pure_logS']).abs()
df['x1_logS1'] = df['x1'] * df['S1_pure_logS']
df['x2_logS2'] = df['x2'] * df['S2_pure_logS']
df['x1x2_logS_diff'] = df['x1'] * df['x2'] * (df['S1_pure_logS'] - df['S2_pure_logS'])
df['x1x2_abs_logS_diff'] = df['x1'] * df['x2'] * (df['S1_pure_logS'] - df['S2_pure_logS']).abs()

# 4. mixture features по дескрипторам растворителей
# предполагается, что в df уже есть столбцы solv1_* и solv2_*

solv1_cols = [c for c in df.columns if c.startswith('solv1_')]
common_pairs = []

for c1 in solv1_cols:
    base = c1.replace('solv1_', '')
    c2 = f'solv2_{base}'
    if c2 in df.columns:
        common_pairs.append((c1, c2, base))

for c1, c2, base in common_pairs:
    df[f'mix_avg_{base}'] = df['x1'] * df[c1] + df['x2'] * df[c2]
    df[f'mix_diff_{base}'] = df[c1] - df[c2]
    df[f'mix_abs_diff_{base}'] = (df[c1] - df[c2]).abs()
    df[f'mix_x1x2_abs_diff_{base}'] = df['x1'] * df['x2'] * (df[c1] - df[c2]).abs()

In [10]:
df.to_csv("MixtureSolDB_with_features.csv", index=False)

print(df.shape)
print("Saved:")
print(" - MixtureSolDB_with_features.csv")

(117302, 127)
Saved:
 - MixtureSolDB_with_features.csv
